# 03 — Pipeline Validation

This notebook validates the complete preprocessing pipeline against the full
selected dataset.

The dataset is processed in chunks to avoid loading the complete dataset into
memory. Statistics are accumulated across all chunks and are used to verify
that the final cleaned dataset satisfies the project requirements.

## Step 1 — Configuration

In [1]:
from pathlib import Path
import json
import pandas as pd

In [2]:
from foresightai.database.connection import get_postgres_connection
from foresightai.database.repositories.review_hash_repository import (
    is_duplicate_hash,
    insert_review_hash,
)
from foresightai.ingestion.preprocessing import (
    preprocess_record,
    preprocess_chunk,
)

In [3]:
print(preprocess_chunk)
print(get_postgres_connection)

<function preprocess_chunk at 0x000002113B1FF740>
<function get_postgres_connection at 0x0000021139EF4360>


In [4]:
conn = get_postgres_connection()

print("PostgreSQL connection successful.")

conn.close()

PostgreSQL connection successful.


In [5]:
# ============================================================
# PATH CONFIGURATION
# ============================================================

RAW_PATH = Path("../../data/raw/Software.jsonl")

PROCESSED_DIR = Path("../../data/processed")
CLEAN_OUTPUT_PATH = PROCESSED_DIR / "software_clean.jsonl"

REPORT_DIR = Path("../../reports")
REPORT_PATH = REPORT_DIR / "03_pipeline_validation_report.md"


# ============================================================
# PROCESSING CONFIGURATION
# ============================================================

CHUNK_SIZE = 5_000

# Minimum number of clean reviews required by the project
MIN_CLEAN_REVIEWS = 10_000


# ============================================================
# CREATE REQUIRED DIRECTORIES
# ============================================================

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)


print("Configuration loaded successfully.")
print(f"Raw dataset      : {RAW_PATH}")
print(f"Chunk size       : {CHUNK_SIZE:,}")
print(f"Minimum reviews  : {MIN_CLEAN_REVIEWS:,}")
print(f"Clean output     : {CLEAN_OUTPUT_PATH}")
print(f"Report output    : {REPORT_PATH}")

Configuration loaded successfully.
Raw dataset      : ..\..\data\raw\Software.jsonl
Chunk size       : 5,000
Minimum reviews  : 10,000
Clean output     : ..\..\data\processed\software_clean.jsonl
Report output    : ..\..\reports\03_pipeline_validation_report.md


## Step 2 - Verifying Input File

In [6]:
if not RAW_PATH.exists():
    raise FileNotFoundError(
        f"Raw dataset not found: {RAW_PATH.resolve()}"
    )

print("Raw dataset found.")
print(f"File size: {RAW_PATH.stat().st_size / (1024**3):.2f} GB")

Raw dataset found.
File size: 1.74 GB


In [7]:
if RAW_PATH.suffix.lower() not in {".jsonl", ".json"}:
    raise ValueError(
        f"Expected a JSON/JSONL dataset, got: {RAW_PATH.suffix}"
    )

print(f"Input format: {RAW_PATH.suffix.lower()}")

Input format: .jsonl


## Step 3 — Build the full-dataset chunk reader

In [8]:
def read_dataset_in_chunks(path: Path, chunk_size: int = 5_000):
    """
    Read a JSON/JSONL dataset incrementally using pandas chunks.

    Parameters
    ----------
    path : Path
        Path to the raw dataset.

    chunk_size : int
        Number of records loaded into memory at a time.

    Yields
    ------
    pd.DataFrame
        One chunk of the dataset at a time.
    """

    if not path.exists():
        raise FileNotFoundError(f"Dataset not found: {path}")

    suffix = path.suffix.lower()

    if suffix in {".jsonl", ".json"}:

        reader = pd.read_json(
            path,
            lines=True,
            chunksize=chunk_size
        )

        for chunk in reader:
            yield chunk

    elif suffix == ".csv":

        reader = pd.read_csv(
            path,
            chunksize=chunk_size
        )

        for chunk in reader:
            yield chunk

    else:
        raise ValueError(
            f"Unsupported dataset format: {suffix}"
        )

## Step 4 — Create the statistics accumulator

In [9]:
def create_statistics_accumulator():
    """
    Create an empty accumulator for full-dataset preprocessing statistics.
    """

    return {
        "input_records": 0,
        "valid_before_deduplication": 0,
        "duplicate_records": 0,
        "clean_records": 0,
        "chunks_processed": 0,
    }

In [10]:
def update_statistics(total_stats, chunk_stats):
    """
    Add statistics from one processed chunk to the cumulative totals.
    """

    total_stats["chunks_processed"] += 1

    for key in [
        "input_records",
        "valid_before_deduplication",
        "duplicate_records",
        "clean_records",
    ]:
        total_stats[key] += chunk_stats.get(key, 0)

    return total_stats

In [11]:
def print_statistics(stats):
    """
    Display accumulated preprocessing statistics.
    """

    print("=" * 50)
    print("PIPELINE VALIDATION STATISTICS")
    print("=" * 50)

    print(f"Chunks processed              : {stats['chunks_processed']:,}")
    print(f"Input records                 : {stats['input_records']:,}")
    print(
        f"Valid before deduplication    : "
        f"{stats['valid_before_deduplication']:,}"
    )
    print(
        f"Duplicate records             : "
        f"{stats['duplicate_records']:,}"
    )
    print(
        f"Clean records                 : "
        f"{stats['clean_records']:,}"
    )

## Step 5 — Test the reader WITHOUT preprocessing

In [12]:
total_rows_read = 0
chunk_count = 0

for chunk in read_dataset_in_chunks(
    RAW_PATH,
    chunk_size=CHUNK_SIZE
):
    
    chunk_count += 1
    total_rows_read += len(chunk)

    if chunk_count <= 3:
        print(
            f"Chunk {chunk_count}: "
            f"{len(chunk):,} records"
        )

print()
print("=" * 50)
print("READER TEST")
print("=" * 50)
print(f"Chunks read : {chunk_count:,}")
print(f"Rows read   : {total_rows_read:,}")

Chunk 1: 5,000 records
Chunk 2: 5,000 records
Chunk 3: 5,000 records

READER TEST
Chunks read : 977
Rows read   : 4,880,181


## Step 6 — Validate the chunk sizes

In [13]:
chunk_sizes = []

for chunk in read_dataset_in_chunks(
    RAW_PATH,
    chunk_size=CHUNK_SIZE
):
    chunk_sizes.append(len(chunk))

print(f"Number of chunks : {len(chunk_sizes):,}")
print(f"First chunk      : {chunk_sizes[0]:,} records")
print(f"Last chunk       : {chunk_sizes[-1]:,} records")
print(f"Total records    : {sum(chunk_sizes):,}")

Number of chunks : 977
First chunk      : 5,000 records
Last chunk       : 181 records
Total records    : 4,880,181


## Step 7 — Initialize the accumulator

In [14]:
total_stats = create_statistics_accumulator()

print_statistics(total_stats)

PIPELINE VALIDATION STATISTICS
Chunks processed              : 0
Input records                 : 0
Valid before deduplication    : 0
Duplicate records             : 0
Clean records                 : 0


## Step 8 — Process the full dataset

### 8.1 First, create a fresh accumulator

In [15]:
total_stats = create_statistics_accumulator()

### 8.2 Run one validation chunk first

In [16]:
# ============================================================
# SINGLE-CHUNK VALIDATION TEST
# ============================================================

print("Single-chunk validation")
print("=" * 50)

# Create a fresh dataset iterator
iter_full_dataset = read_dataset_in_chunks(
    RAW_PATH,
    chunk_size=CHUNK_SIZE
)

# Get the first chunk
first_chunk = next(iter_full_dataset)

print(f"Input chunk size: {len(first_chunk):,}")

Single-chunk validation
Input chunk size: 5,000


In [17]:
TEXT_COLUMN = "text"
RATING_COLUMN = "rating"

conn = get_postgres_connection()

try:
    clean_chunk, stats = preprocess_chunk(
        chunk=first_chunk,
        text_column=TEXT_COLUMN,
        rating_column=RATING_COLUMN,
        connection=conn
    )

    conn.commit()

finally:
    conn.close()

In [18]:
print("\nSingle-chunk statistics")
print("=" * 50)

for key, value in stats.items():
    print(f"{key}: {value:,}")

print("\nClean records:")
print(clean_chunk.head())


Single-chunk statistics
non_english: 511
duplicate_records: 78
empty_or_too_short: 85
undetectable_language: 2
input_records: 5,000
valid_before_deduplication: 4,402
clean_records: 4,324

Clean records:
   rating           title                                               text  \
0       4  Good  delivery                                  Good for the kids   
1       3   Good delivery                                    Ok for the kids   
2       3          CS 2.0      ok game!!!! end up not playing that much.....   
3       1        One Star         hate this game, too hard for children.....   
4       2       Two Stars  it was ok kids had hard time knowing what to d...   

  images        asin parent_asin                       user_id  \
0     []  B00LV4D70O  B00LV4D70O  AHFZUNQFXSVVT6Z6BYKE5CLBX3KQ   
1     []  B00PSGW79I  B00PSGW79I  AHFZUNQFXSVVT6Z6BYKE5CLBX3KQ   
2     []  B00A4EZ3QS  B00A4EZ3QS  AHFZUNQFXSVVT6Z6BYKE5CLBX3KQ   
3     []  B013OU7XES  B013OU7XES  AHFZUNQFXSVVT6Z6B

In [19]:
conn = get_postgres_connection()

try:
    with conn.cursor() as cur:
        cur.execute("TRUNCATE TABLE review_dedupe;")

    conn.commit()

finally:
    conn.close()